In [56]:
import json
import pandas as pd
import datasets

In [ ]:
HF_DATASET_NAME = "matsant01/blind-spots-bench"

288

In [ ]:
df = pd.read_excel("../data/reasoning_course.xlsx", sheet_name="dataset-0319")
print("Original length:", len(df))

# Filter invalid questions
print(f"Removing {len(df[df['valid_flag'] != 1])} invalid questions.")
df = df[df['valid_flag'] == 1].copy()
print(f"Remaining valid questions: {len(df)}")

Removing 52 invalid questions.
Remaining valid questions: 236


In [ ]:
# Create a map from 'author' name to hash
import hashlib
unique_authors = df['author'].dropna().unique()
author_map = {author: hashlib.sha256(str(author).encode()).hexdigest()[:8] for author in unique_authors}

# Replace author names with their hashes in the dataframe
df['author'] = df['author'].map(author_map)

# Save the author map to a JSON file in the data/ directory
# import json
# with open("../data/author_map.json", "w") as f:
#     json.dump(author_map, f, indent=4)

In [60]:
import os
import shutil


for index, row in df.iterrows():
    if row["question_type"] in ["multi-to-image", "multi-to-text"]:
        if pd.isna(row['img_path']):
            print(f"Error: Missing image path for QID {row['QID']}")
            continue
        else:
            # Check the existence of the image file
            full_img_path = os.path.join("../data/images/", row['img_path'])
            if os.path.isfile(full_img_path):
                # Overwrite the complete image path in the dataframe
                df.at[index, 'img_path'] = full_img_path
            else:
                print(f"Error: Image file not found for QID {row['QID']} at path {row['img_path']}")
    else:
        if not pd.isna(row['img_path']):
            print(f"Warning: Question {row['QID']} of type '{row['question_type']}' should not have an image path, but found '{row['img_path']}'.")

In [61]:
df.drop(columns=["missing_image", "sheet", "valid_reason", "valid_flag", "comments", "percentages", "occurances", "Types"], inplace=True)
df.rename(columns={"sub-task type": "sub-task_type", "Failure mode": "failure_mode"}, inplace=True)
df["Re_Eval"] = df["Re_Eval"].fillna(0.0).astype(bool)
df["QID"] = df.apply(lambda row: row["QID"] + 1000 if row["Re_Eval"] else row['QID'], axis=1).astype(int)

In [62]:
# Reduce size of images to max 1024x1024 while maintaining aspect ratio
from PIL import Image

max_size = (1024, 1024)

for index, row in df.iterrows():
    if pd.notna(row['img_path']):
        try:
            with Image.open(row['img_path']) as img:
                # Check if resize is needed
                if img.size[0] > max_size[0] or img.size[1] > max_size[1]:
                    img.thumbnail(max_size)
                    img.save(row['img_path'])
        except Exception as e:
            print(f"Error processing image at '{row['img_path']}' (question {row['QID']}): {e}")

In [63]:
def clean_link_field(link):
	values_to_remove = [
		"(Gemini 2.5 Flash)",
		"(ChatGPT 5)",
		"(Gemini 2.5 Pro)\n",
		" (Gemini Pro 2.5)\n",
		" (Gemini 2.5 Flash)\n",
	]
	for v in values_to_remove:
		if v in link:
			link = link.split(v)[0]

	return link.strip()


df['link'] = df['link'].apply(clean_link_field)

# Drop rows where both 'prompt' and 'image' are missing
initial_len = len(df)
df = df[~(df['prompt'].isna() & df['img_path'].isna())]
print(f"Removed {initial_len - len(df)} rows with both 'prompt' and 'image' missing")
df['prompt'] = df['prompt'].fillna('')

Removed 0 rows with both 'prompt' and 'image' missing


In [64]:
# Create a Dataset from the DataFrame and push to huggingface hub
from datasets import Dataset, Image, DatasetDict

# Rename img_path to image for standard naming
if 'img_path' in df.columns:
    df = df.rename(columns={'img_path': 'image'})

def get_image_path(path):
    if isinstance(path, str) and os.path.exists(path):
        return path
    return None

df['image'] = df['image'].apply(get_image_path)

### Split main dataset and re-eval dataset

In [65]:
# Keep in the delta dataset only the problem with "Re_Eval" set
delta_df = df[df["Re_Eval"] == True]

# Drop the column
delta_df.drop(columns=["Re_Eval"], inplace=True)
df.drop(columns=["Re_Eval"], inplace=True)


# Create main dataset
ds = Dataset.from_pandas(df, preserve_index=False)
# Cast the image column to Image feature
ds = ds.cast_column("image", Image())
df_dict = DatasetDict({"test": ds})

# Create delta dataset
delta_ds = Dataset.from_pandas(delta_df, preserve_index=False)
delta_ds = delta_ds.cast_column("image", Image())
delta_df_dict = DatasetDict({"test": delta_ds})


print(ds)
print(delta_ds)

Dataset({
    features: ['QID', 'prompt', 'solution', 'question_type', 'task_type', 'sub-task_type', 'multi_subcategory_flag', 'failure_mode', 'link', 'author', 'image'],
    num_rows: 236
})
Dataset({
    features: ['QID', 'prompt', 'solution', 'question_type', 'task_type', 'sub-task_type', 'multi_subcategory_flag', 'failure_mode', 'link', 'author', 'image'],
    num_rows: 12
})


/var/folders/vd/v1_75h5d54qdztdtkk73k1l80000gn/T/ipykernel_73046/3469957695.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  delta_df.drop(columns=["Re_Eval"], inplace=True)


### Show a diff before pushing

In [ ]:
old_ds = datasets.load_dataset(f"{HF_DATASET_NAME}", split="test")

In [67]:
df = ds.to_pandas()
old_df = old_ds.to_pandas()

# Merge
merged = pd.merge(df, old_df, on="QID", how="inner", suffixes=('_new', '_old'), indicator=True)
print(f"New: {len(df)} - Old: {len(old_df)} - Merged: {len(merged)}")

# Check for differences in the merged dataset
for i, row in merged.iterrows():
	has_diff = False
	for col in df.columns:
		if col == "image":
			continue
		new_col = f"{col}_new"
		old_col = f"{col}_old"
		if new_col in merged.columns and old_col in merged.columns:
			if row[new_col] != row[old_col]:
				if not has_diff:
					print(f"Difference in QID {row['QID']}:")
					has_diff = True
				print(f"  Column: {col}\t {row[old_col]} -> {row[new_col]}")
	if has_diff:
		print()

New: 236 - Old: 236 - Merged: 236
Difference in QID 122:
  Column: sub-task_type	 perceptual counting, spatial reasoning -> perceptual counting
  Column: multi_subcategory_flag	 1.0 -> 0.0



### Check distribution before pushing

In [68]:
print(df["question_type"].value_counts())
print(delta_df["question_type"].value_counts())

question_type
text-only         109
text-to-image      76
multi-to-text      43
multi-to-image      7
text-to-multi       1
Name: count, dtype: int64
question_type
text-only        9
text-to-image    3
Name: count, dtype: int64


## Push to hub

In [ ]:
# Push to hub
df_dict.push_to_hub(f"{HF_DATASET_NAME}")

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 38.45ba/s]
Processing Files (1 / 1): 100%|██████████| 11.5MB / 11.5MB, 90.2kB/s  
New Data Upload: 100%|██████████| 90.1kB / 90.1kB, 90.2kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:02<00:00,  2.46s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/matsant01/blind-spots-bench/commit/065abe7297d0c15d4e190b2b9b69b7678a19813b', commit_message='Upload dataset', commit_description='', oid='065abe7297d0c15d4e190b2b9b69b7678a19813b', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/matsant01/blind-spots-bench', endpoint='https://huggingface.co', repo_type='dataset', repo_id='matsant01/blind-spots-bench'), pr_revision=None, pr_num=None)

In [ ]:
delta_df_dict.push_to_hub(f"{HF_DATASET_NAME}-delta", private=True)

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 859.84ba/s]
Processing Files (1 / 1): 100%|██████████| 14.5kB / 14.5kB, 36.3kB/s  
New Data Upload: 100%|██████████| 14.5kB / 14.5kB, 36.3kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,  1.06s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/matsant01/blind-spots-bench-delta/commit/191b65eaa1211f0bdc94b339167cd6685adf4fd1', commit_message='Upload dataset', commit_description='', oid='191b65eaa1211f0bdc94b339167cd6685adf4fd1', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/matsant01/blind-spots-bench-delta', endpoint='https://huggingface.co', repo_type='dataset', repo_id='matsant01/blind-spots-bench-delta'), pr_revision=None, pr_num=None)